In [12]:
# Autoload modules
%load_ext autoreload
%autoreload 2

In [14]:
from qiskit_algorithms.utils import algorithm_globals
from qiskit.quantum_info import Pauli, PauliList, SparsePauliOp
import numpy as np
import sys
sys.path.append('../')

In [9]:
chain_len = 4
pair_energies = algorithm_globals.random.random(
    (chain_len + 1, 2, chain_len + 1, 2)
)

In [10]:
pair_energies.shape

(5, 2, 5, 2)

In [47]:
pauli_1 = Pauli('ZIZZIII')
pauli_2 = Pauli('IZZZIZI')

In [48]:
sp_op = 0.1 * SparsePauliOp([pauli_1]) + 0.3 * SparsePauliOp([pauli_2])
sp_op

SparsePauliOp(['ZIZZIII', 'IZZZIZI'],
              coeffs=[0.1+0.j, 0.3+0.j])

In [53]:
for hamiltonian in sp_op:
    print(hamiltonian.coeffs[0])
    table_z = np.copy(hamiltonian.paulis.z[0])

(0.1+0j)
(0.3+0j)


In [55]:
def _calc_updated_coeffs(
    hamiltonian: SparsePauliOp, table_z, has_side_chain_second_bead: bool
) -> np.ndarray:
    coeffs = np.copy(hamiltonian.coeffs[0])
    if len(table_z) > 1 and table_z[1] == np.bool_(True):
        coeffs = -1 * coeffs
    if (
        not has_side_chain_second_bead
        and len(table_z) > 6
        and table_z[5] == np.bool_(True)
    ):
        coeffs = -1 * coeffs
    return coeffs

def _preset_binary_vals(table_z, has_side_chain_second_bead: bool):
    main_beads_indices = [0, 1, 2, 3]
    if not has_side_chain_second_bead:
        main_beads_indices.append(5)
    for index in main_beads_indices:
        _preset_single_binary_val(table_z, index)


def _preset_single_binary_val(table_z, index: int):
    try:
        table_z[index] = np.bool_(False)
    except IndexError:
        pass

In [57]:
new_tables_x = []
new_tables_z = []
new_coeffs = []
has_side_chain_second_bead = False
for hamiltonian in sp_op:
    table_z = np.copy(hamiltonian.paulis.z[0])
    table_x = np.copy(hamiltonian.paulis.x[0])
    coeffs = _calc_updated_coeffs(hamiltonian, table_z, has_side_chain_second_bead)
    _preset_binary_vals(table_z, has_side_chain_second_bead)
    new_tables_x.append(table_x)
    new_tables_z.append(table_z)
    new_coeffs.append(coeffs)
new_pauli_table = PauliList.from_symplectic(new_tables_z, new_tables_x)
operator_updated = SparsePauliOp(data=new_pauli_table, coeffs=new_coeffs)
operator_updated = operator_updated.simplify()

In [60]:
print(new_tables_z)
print(new_tables_x)

[array([False, False, False, False,  True, False,  True]), array([False, False, False, False,  True, False, False])]
[array([False, False, False, False, False, False, False]), array([False, False, False, False, False, False, False])]


In [61]:
PauliList.from_symplectic(new_tables_z, new_tables_x)

PauliList(['ZIZIIII', 'IIZIIII'])

In [58]:
operator_updated

SparsePauliOp(['ZIZIIII', 'IIZIIII'],
              coeffs=[0.1+0.j, 0.3+0.j])

In [45]:
table_z

array([ True,  True,  True, False])

In [24]:
sp_op.paulis

PauliList(['ZIZZ', 'IZZZ'])

In [40]:
(0.5 * sp_op - 0.1 * sp_op).simplify()

SparsePauliOp(['ZZII'],
              coeffs=[0.4+0.j])

In [29]:
SparsePauliOp("I" * chain_len)

SparsePauliOp(['IIII'],
              coeffs=[1.+0.j])

In [64]:
sp_op.num_qubits

7

In [74]:
from qufold.qubit_utils.qubit_number_reducer import remove_unused_qubits

In [73]:
2 * sp_op 

SparsePauliOp(['ZIZZIII', 'IZZZIZI'],
              coeffs=[0.2+0.j, 0.6+0.j])

In [70]:
remove_unused_qubits(sp_op)

(SparsePauliOp(['ZIZZI', 'IZZZZ'],
               coeffs=[0.1+0.j, 0.3+0.j]),
 [0, 2])

In [75]:
sp_op + 0

SparsePauliOp(['ZIZZIII', 'IZZZIZI'],
              coeffs=[0.1+0.j, 0.3+0.j])

In [76]:
"Z" + "I" * 3

'ZIII'

In [77]:
"I" * 3 + "Z"

'IIIZ'

In [2]:
from qiskit.opflow import PauliOp, I, Z


def _build_full_identity(num_qubits: int) -> PauliOp:
    """
    Builds a full identity operator of a given size.

    Args:
        num_qubits: number of qubits on which a full identity operator will be created.

    Returns:
        A full identity operator of a given size.
    """
    full_identity = I
    for _ in range(1, num_qubits):
        full_identity = I ^ full_identity
    return full_identity


def _build_pauli_z_op(num_qubits: int, pauli_z_indices) -> PauliOp:
    """
    Builds a Pauli operator of a given size with Pauli Z operators on indicated positions and
    identity operators on other positions.

    Args:
        num_qubits: number of qubits on which a Pauli operator will be created.
        pauli_z_indices: a set of indices in a Pauli operator on which a Pauli Z operator shall
                        appear.

    Returns:
        A Pauli operator of a given size with Pauli Z operators on indicated positions and
        identity operators on other positions.
    """
    if 0 in pauli_z_indices:
        operator = Z
    else:
        operator = I
    for i in range(1, num_qubits):
        if i in pauli_z_indices:
            operator = Z ^ operator
        else:
            operator = I ^ operator

    return operator

In [5]:
_build_pauli_z_op(5, [0, 3])

PauliOp(Pauli('IZIIZ'), coeff=1.0)

In [2]:
from qiskit.quantum_info import SparsePauliOp


def _build_full_identity(num_qubits: int) -> SparsePauliOp:
    """
    Builds a full identity operator of a given size.

    Args:
        num_qubits: number of qubits on which a full identity operator will be created.

    Returns:
        A full identity operator of a given size.
    """
    full_identity = SparsePauliOp("I" * num_qubits)
    return full_identity


def _build_pauli_z_op(num_qubits: int, pauli_z_indices) -> SparsePauliOp:
    """
    Builds a Pauli operator of a given size with Pauli Z operators on indicated
    positions and identity operators on other positions.

    Args:
        num_qubits: number of qubits on which a Pauli operator will be created.
        pauli_z_indices: a set of indices in a Pauli operator on which a Pauli Z
        operator shall appear.

    Returns:
        A Pauli operator of a given size with Pauli Z operators on indicated
        positions and identity operators on other positions.
    """
    if 0 in pauli_z_indices:
        operator_str = "Z"
    else:
        operator_str = "I"
    for i in range(1, num_qubits):
        if i in pauli_z_indices:
            operator_str = "Z" + operator_str
        else:
            operator_str = "I" + operator_str
    operator = SparsePauliOp(operator_str)
    return operator

In [3]:
_build_pauli_z_op(5, [0, 3])

SparsePauliOp(['IZIIZ'],
              coeffs=[1.+0.j])

In [1]:
# Opflow test
from qiskit.opflow import PauliOp, I, Z, X
po1 = X ^ I ^ Z^ I
po2 = I ^ X ^ Z ^ I
print(po1 @ po2)
print(po1 ^ po2)
print(2 * po1 + 3 * po2) 

XXII
XIZIIXZI
2.0 * XIZI
+ 3.0 * IXZI


/var/folders/6z/68dj05v5057cwh6yv32zx6380000gp/T/ipykernel_3347/3913223756.py:2: DeprecationWarning: The ``qiskit.opflow`` module is deprecated as of qiskit-terra 0.24.0. It will be removed in Qiskit 1.0. For code migration guidelines, visit https://qisk.it/opflow_migration.
  from qiskit.opflow import PauliOp, I, Z, X


In [1]:
# quantum_info test
from qiskit.quantum_info import SparsePauliOp
sp1 = SparsePauliOp("XIZI")
sp2 = SparsePauliOp("IXZI")
print(sp1 @ sp2)
print(sp1 ^ sp2)
print(2 * sp1 + 3 * sp2)

SparsePauliOp(['XXII'],
              coeffs=[1.+0.j])
SparsePauliOp(['XIZIIXZI'],
              coeffs=[1.+0.j])
SparsePauliOp(['XIZI', 'IXZI'],
              coeffs=[2.+0.j, 3.+0.j])
